### 📚 Importing Libraries

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from utils.helpers import *
from utils.models import *
from utils.params import *

# Setting the seed for reproducibility
setSeed()

# embeddings_folder = dataset_folder.replace("datasets", "align")
# embeddings_folder = dataset_folder.replace("datasets", "blip")
# embeddings_folder = dataset_folder.replace("datasets", "dinotxt")
embeddings_folder = dataset_folder.replace("datasets", "embeddings")
# embeddings_folder = dataset_folder.replace("datasets", "openclip")
# embeddings_folder = dataset_folder.replace("datasets", "siglip")

### 💻 Loading datasets

The datasets are the following:

* **`JB28K`**: contains multimodal jailbreaks and redteam queries;
* **`VQA`**: contains multimodal beniqn queries.
* **`Wayve`**: contains multimodal beniqn queries, from which we previously generated redteam queries;
* **`NuScenes`**: although it contains scene descriptions, we previously used it to generate RoboPAIR jailbreaks.

In [ ]:
jb28k = [get_embedding('jb28k', embeddings_folder)]
vqa = [get_embedding('vqa', embeddings_folder)]
wayve = [get_embedding('wayve', embeddings_folder)]
nuscenes = [get_embedding('nuscenes', embeddings_folder)]

### 0️⃣ Baseline Classifier

To provide a baseline for our results, we train a classifier on general-purpose jailbreak and benign dataset and we test its generalization capabilities on our domain-specific dataset. Also, we only load text and image embeddings, without performing our multimodal fusion technique.

#### 0️⃣.1️⃣ Text only

We train and test a classifier with only CLIP embeddings generated from text, without domain adaptation.

In [ ]:
# Loading datasets
with status("Loading datasets"):
    X_general, y_general, domain_general = load_embeddings(jb28k + vqa, domain_label=0, mode='text')
    X_domain, y_domain, domain_domain = load_embeddings(wayve + nuscenes, domain_label=1, mode='text')

    # Shuffle X_general, y_general, domain_general
    rng = np.random.default_rng(seed)
    indices = rng.permutation(X_general.shape[0])
    X_general = X_general[indices]
    y_general = y_general[indices]
    domain_general = domain_general[indices]

    # Shuffle X_domain, y_domain, domain_domain
    indices = rng.permutation(X_domain.shape[0])
    X_domain = X_domain[indices]
    y_domain = y_domain[indices]
    domain_domain = domain_domain[indices]

    # Ensuring consistent number of samples
    min_samples = min(X_general.shape[0], X_domain.shape[0])
    X_general, y_general, domain_general = X_general[:min_samples], y_general[:min_samples], domain_general[:min_samples]
    X_domain, y_domain, domain_domain = X_domain[:min_samples], y_domain[:min_samples], domain_domain[:min_samples]

    # Defining train and test sets
    X_train = X_general
    y_train = y_general
    X_test = X_domain
    y_test = y_domain

    # Converting to PyTorch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)

    # DataLoader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Defining the domain classifier
model = SimpleNN(X_train.shape[1], 2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"[🔁 EPOCH {epoch+1}/{num_epochs}] Loss: {total_loss/len(train_loader):.8f}")

In [ ]:
# Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model(batch_X)
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

baseline_text_accuracy = 100 * correct / total
print(f"[🔍 BASELINE CLASSIFIER] Accuracy: {baseline_text_accuracy:.2f}%")

#### 0️⃣.2️⃣ Text + Image

We train and test a classifier with concatenated text and image CLIP embeddings, without domain adaptation.

In [ ]:
# Loading datasets
with status("Loading datasets"):
    X_general, y_general, domain_general = load_embeddings(jb28k + vqa, domain_label=0, mode='nofused')
    X_domain, y_domain, domain_domain = load_embeddings(wayve + nuscenes, domain_label=1, mode='nofused')

    # Shuffle X_general, y_general, domain_general
    rng = np.random.default_rng(seed)
    indices = rng.permutation(X_general.shape[0])
    X_general = X_general[indices]
    y_general = y_general[indices]
    domain_general = domain_general[indices]

    # Shuffle X_domain, y_domain, domain_domain
    indices = rng.permutation(X_domain.shape[0])
    X_domain = X_domain[indices]
    y_domain = y_domain[indices]
    domain_domain = domain_domain[indices]

    # Ensuring consistent number of samples
    min_samples = min(X_general.shape[0], X_domain.shape[0])
    X_general, y_general, domain_general = X_general[:min_samples], y_general[:min_samples], domain_general[:min_samples]
    X_domain, y_domain, domain_domain = X_domain[:min_samples], y_domain[:min_samples], domain_domain[:min_samples]

    # Defining train and test sets
    X_train = X_general
    y_train = y_general
    X_test = X_domain
    y_test = y_domain

    # Converting to PyTorch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)

    # DataLoader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Defining the domain classifier
model = SimpleNN(X_train.shape[1], 2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"[🔁 EPOCH {epoch+1}/{num_epochs}] Loss: {total_loss/len(train_loader):.8f}")

In [ ]:
# Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model(batch_X)
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

baseline_nofused_accuracy = 100 * correct / total
print(f"[🔍 BASELINE CLASSIFIER] Accuracy: {baseline_nofused_accuracy:.2f}%")

#### 1️⃣ Domain Classification

First, we train a classifier to distinguish between the general datasets and the domain-specific datasets.

In [ ]:
# Loading datasets
with status("Loading datasets"):
    X_general, y_general, domain_general = load_embeddings(jb28k + vqa, domain_label=0, mode='full')
    X_domain, y_domain, domain_domain = load_embeddings(wayve + nuscenes, domain_label=1, mode='full')

    # Ensuring consistent number of samples
    min_samples = min(X_general.shape[0], X_domain.shape[0])
    X_general, y_general, domain_general = X_general[:min_samples], y_general[:min_samples], domain_general[:min_samples]
    X_domain, y_domain, domain_domain = X_domain[:min_samples], y_domain[:min_samples], domain_domain[:min_samples]

    # Combining for domain classification
    X_domain_cls = np.vstack((X_general, X_domain))
    y_domain_cls = np.hstack((domain_general, domain_domain))

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X_domain_cls, y_domain_cls, test_size=0.2, random_state=seed)

    # Undersampling to balance classes in train and test sets
    rus = RandomUnderSampler(random_state=seed)
    X_train, y_train = rus.fit_resample(X_train, y_train)
    X_test, y_test = rus.fit_resample(X_test, y_test)

    # Converting to PyTorch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)

    # DataLoader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Defining the domain classifier
model = SimpleNN(X_train.shape[1], 2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"[🔁 EPOCH {epoch+1}/{num_epochs}] Loss: {total_loss/len(train_loader):.8f}")

In [ ]:
# Evaluation
model.eval()
with torch.no_grad():
    prob_general = torch.softmax(model(torch.tensor(X_general, dtype=torch.float32).to(device)), dim=1)[:, 1]
    prob_domain = torch.softmax(model(torch.tensor(X_domain, dtype=torch.float32).to(device)), dim=1)[:, 1]

    # Performing evaluation on test set
    model.eval()
    with torch.no_grad():
        test_outputs = model(torch.tensor(X_test, dtype=torch.float32).to(device))
        test_probabilities = torch.softmax(test_outputs, dim=1)[:, 1]
        test_predictions = torch.argmax(test_outputs, dim=1)
        domain_classifier_accuracy = (test_predictions.cpu().numpy() == y_test).mean()

    print(f"[🔍 DOMAIN CLASSIFIER] Accuracy: {domain_classifier_accuracy * 100:.2f}%")

In [ ]:
# Extracting weights probabilities
weights = prob_domain / prob_general

# Normalizing weights
weights = weights.cpu().numpy()
weights /= np.sum(weights)

#### 2️⃣ Non-Adapted Classifier

To compare our methodology with traditional methods, we train a classifier without performing domain adaptation, i.e., by training on general + benign domain-specific datasets, and testing on domain-specific jailbreak datasets.

In [ ]:
# Loading datasets
with status("Loading datasets"):
    X1, y1, d1 = load_embeddings(jb28k + vqa, domain_label=0, mode='full')
    X2, y2, d2 = load_embeddings(wayve, domain_label=1, mode='full')

    X_general = np.concatenate([X1, X2])
    y_general = np.concatenate([y1, y2])
    domain_general = np.concatenate([d1, d2])

    X_t, y_t, domain_t = load_embeddings(nuscenes, domain_label=1, label='bq+jb', mode='full')

    X_train, y_train = X_general, y_general
    X_test, y_test = X_t, y_t

    # Undersampling to balance classes in train set
    X_train, y_train = rus.fit_resample(X_train, y_train)

    # Converting to tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)

    # DataLoader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Defining the jailbreak classifier
model = SimpleNN(X_train.shape[1], 2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"[🔁 EPOCH {epoch+1}/{num_epochs}] Loss: {total_loss/len(train_loader):.8f}")

In [ ]:
# Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model(batch_X)
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

non_adapted_accuracy = 100 * correct / total
print(f"[🔍 NON-ADAPTED CLASSIFIER] Accuracy: {non_adapted_accuracy:.2f}%")

#### 3️⃣ Adapted Classifier

We now perform our domain adaptation methodology on the fused embeddings. Note that we will use weights that have been computed after domain classification.

In [ ]:
# Computing CORAL distance
coral_value = compute_coral(X_general, X_domain)
print(f"[🪸 CORAL] General vs Domain: {coral_value:.2f}")

In [ ]:
# Loading datasets
with status("Loading datasets"):
    # Ensuring consistent number of samples
    indices_general = np.random.choice(X_general.shape[0], min_samples, replace=False)
    X_general, y_general, domain_general = X_general[indices_general], y_general[indices_general], domain_general[indices_general]

    # Adaptation
    X_general = coral_adaptation(X_general, X_domain)

    X_train, y_train = X_general, y_general
    X_test, y_test = X_t, y_t

    # Convert to tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)

    # DataLoader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Defining the adapted jailbreak classifier
model = SimpleNN(X_train.shape[1], 2).to(device)
criterion = WeightedLoss(nn.CrossEntropyLoss())
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for i, (batch_X, batch_y) in enumerate(train_loader):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        batch_weights = torch.tensor(weights[i * batch_X.shape[0]:(i + 1) * batch_X.shape[0]], dtype=torch.float32).to(device)
        loss = criterion(outputs, batch_y, batch_weights)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"[🔁 EPOCH {epoch+1}/{num_epochs}] Loss: {total_loss/len(train_loader):.8f}")

In [ ]:
# Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

adapted_accuracy = correct / total
print(f"[🔍 ADAPTED CLASSIFIER] Accuracy: {adapted_accuracy*100:.2f}%")

In [ ]:
torch.save(model.state_dict(), "./models/car_classifier.pth")

#### 3️⃣.1️⃣ Ablation

We now train a classifier on non-fused concatenated text and image CLIP embeddings with our domain adaptation methodology.

In [ ]:
# Loading datasets
with status("Loading datasets"):
    X_general, y_general, domain_general = load_embeddings(jb28k + vqa, domain_label=0, mode='nofused')
    X_domain, y_domain, domain_domain = load_embeddings(wayve + nuscenes, domain_label=1, mode='nofused')

    # Shuffle X_general, y_general, domain_general
    rng = np.random.default_rng(seed)
    indices = rng.permutation(X_general.shape[0])
    X_general = X_general[indices]
    y_general = y_general[indices]
    domain_general = domain_general[indices]

    # Shuffle X_domain, y_domain, domain_domain
    indices = rng.permutation(X_domain.shape[0])
    X_domain = X_domain[indices]
    y_domain = y_domain[indices]
    domain_domain = domain_domain[indices]

    # Ensuring consistent number of samples
    min_samples = min(X_general.shape[0], X_domain.shape[0])
    X_general, y_general, domain_general = X_general[:min_samples], y_general[:min_samples], domain_general[:min_samples]
    X_domain, y_domain, domain_domain = X_domain[:min_samples], y_domain[:min_samples], domain_domain[:min_samples]

    # Defining train and test sets
    X_train = X_general
    y_train = y_general
    X_test = X_domain
    y_test = y_domain

    X_general = coral_adaptation(X_general, X_domain)

    # Converting to PyTorch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)

    # DataLoader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Defining the adapted jailbreak classifier
model = SimpleNN(X_train.shape[1], 2).to(device)
criterion = WeightedLoss(nn.CrossEntropyLoss())
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for i, (batch_X, batch_y) in enumerate(train_loader):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        batch_weights = torch.tensor(weights[i * batch_X.shape[0]:(i + 1) * batch_X.shape[0]], dtype=torch.float32).to(device)
        loss = criterion(outputs, batch_y, batch_weights)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"[🔁 EPOCH {epoch+1}/{num_epochs}] Loss: {total_loss/len(train_loader):.8f}")

In [ ]:
# Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

ablated_accuracy = correct / total
print(f"[🔍 ABLATED CLASSIFIER] Accuracy: {ablated_accuracy*100:.2f}%")